# Phase 1+ Belief Geometry Analysis

Computational mechanics analysis of two converged grokking models, replicating
Poncini (2025), Section 5. Pipeline:

1. **Phase 1 — Transformer**: fit (α_k, ω_k) to logits; decode the EHMM
   predictive vectors ⟨v^(·)| at the four residual-stream positions; control
   against the HMM simplex vectors ⟨e_p^(·)|.
2. **Phase 2 — MLP**: same procedure at the pre-ReLU (M1) and post-ReLU (M2)
   hidden states.
3. **Phase 3 — Cross-architecture comparison**: do both architectures pick up
   the EHMM (same algorithm family)? Same Fourier modes or different?
4. **Training trajectory**: replicate Poncini Fig 2 — polygon (EHMM) vs
   simplex (HMM) decoding R² across all training checkpoints. EHMM modes are
   fitted once at the final checkpoint and held fixed across the sweep. Trajectories
   are computed at the post-MLP layer (POS4 / M2) and additionally at the
   pre-MLP layer (POS3 / M1) for a cleaner cross-architecture comparison.

All decoding logic lives in `decoding/` and `analysis/`; this notebook just
loads the trained models and orchestrates the runs.

Figures are saved to `figures/belief_geometry_figs/` as well as displayed inline.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt

from training.config import set_global_seeds
from training.models import GrokTransformer, ChughtaiMLP
from training.checkpoints import load_final_checkpoint
from training.loop import DEFAULT_RUN_ROOT

from analysis.transformer_belief import (
    run_transformer_phase1, summarize_phase1,
    plot_phase1_summary,
    plot_per_frequency_grid as plot_t_per_freq_grid,
    plot_per_frequency_pairs as plot_t_per_freq_pairs,
)
from analysis.mlp_belief import (
    run_mlp_phase2, summarize_phase2,
    plot_phase2_summary, plot_phase2_summary_classmean,
    plot_per_frequency_grid as plot_m_per_freq_grid,
    plot_per_frequency_pairs as plot_m_per_freq_pairs,
)
from analysis.compare import (
    summarize_phase3, plot_alpha_comparison, plot_decoding_comparison,
)
from analysis.trajectory import (
    sweep_transformer, sweep_mlp,
    plot_trajectory, plot_trajectory_comparison,
)

set_global_seeds(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_ROOT = DEFAULT_RUN_ROOT
print('Using device:', DEVICE)

FIG_DIR = os.path.join('figures', 'belief_geometry_figs')
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig_or_ax, name, dpi=150):
    # Save the figure to FIG_DIR; return it so the cell still displays it inline.
    fig = fig_or_ax if hasattr(fig_or_ax, 'savefig') else fig_or_ax.figure
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, bbox_inches='tight', dpi=dpi)
    print(f'Saved {path}')
    return fig

In [ ]:
# Load the converged transformer and MLP from the latest checkpoints.

transformer_config, transformer_ckpt, transformer_metrics = load_final_checkpoint(
    'transformer_run', run_root=RUN_ROOT, device=DEVICE
)
transformer = GrokTransformer(transformer_config).to(DEVICE)
transformer.load_state_dict(transformer_ckpt['model_state'])
transformer.eval()

mlp_config, mlp_ckpt, mlp_metrics = load_final_checkpoint(
    'mlp_run', run_root=RUN_ROOT, device=DEVICE
)
mlp = ChughtaiMLP(mlp_config.p, d_hidden=mlp_config.mlp_d_hidden).to(DEVICE)
mlp.load_state_dict(mlp_ckpt['model_state'])
mlp.eval()

print('Transformer final test accuracy:', transformer_metrics['test_acc'][-1])
print('MLP        final test accuracy:', mlp_metrics['test_acc'][-1])

## Phase 1 — Transformer belief geometry

Targets per Poncini at p=113:
- Logit-fit R² ≥ 0.99 with ~5 frequencies.
- Supervised decoding R² ≈ 0.99 at POS1, POS2, POS3, POS4.
- POS4 simplex control R² ≈ 0.09 (model represents EHMM, not HMM).

The exact (ω_k, α_k) are seed-dependent; only the structure should match.

**Note on `n_pcs`**: the unweighted EHMM target at each position is 2·n_freq
dimensional (here 20-D for 10 frequencies), so the semi-supervised PCA→ridge
fit needs `n_pcs ≥ 20` to potentially saturate. We use `n_pcs=30` for headroom;
per-frequency plots only need 2-D targets so they remain unaffected.

In [ ]:
t_result = run_transformer_phase1(
    transformer, transformer_config.p,
    target_r2=0.99, max_freqs=4, n_pcs=30, ridge_alpha=1e-3,
)
summarize_phase1(t_result)

In [ ]:
save_fig(plot_phase1_summary(t_result), 'phase1_summary.png');

In [ ]:
save_fig(plot_t_per_freq_grid(t_result), 'phase1_per_freq_grid.png');

### Per-frequency decompositions (Poncini Figs 3–5)

Each row = one fitted ω_k. Left column = theoretical predictive vector (unit
circle, colored by relevant index). Right column = PCA→ridge fitted prediction
of that target from the activations at the position. R² annotations show how
faithfully each frequency is recovered. Look for:

- **Ring tightness on the right column** — high-α frequencies give clean rings;
  low-α frequencies trail off into noise.
- **"Polygon width"** at POS4 — Poncini's observation that the post-MLP rings
  aren't infinitely thin. This is attributed to how the MLP performs the
  modular sum.

In [ ]:
# POS4 (post-MLP) — Poncini Fig 3 analogue
save_fig(plot_t_per_freq_pairs(t_result, 'POS4', n_pcs=10), 'phase1_per_freq_pairs_pos4.png');

In [ ]:
# POS2 (post-embed @ b position) — Poncini Fig 4 analogue
save_fig(plot_t_per_freq_pairs(t_result, 'POS2', n_pcs=10), 'phase1_per_freq_pairs_pos2.png');

In [ ]:
# POS3 (post-attention) — Poncini Fig 5 analogue (sum-of-circles target)
save_fig(plot_t_per_freq_pairs(t_result, 'POS3', n_pcs=10), 'phase1_per_freq_pairs_pos3.png');

## Phase 2 — MLP belief geometry

Same procedure on the MLP. Different (ω_k, α_k) are expected (Chughtai weak
universality). M1 = pre-ReLU = `W_a·a + W_b·b` is linear in the indices, so M1
supervised R² should be close to 1 on the additive target ⟨v^(a)| + ⟨v^(b)|.
The interesting comparison is M2 (post-ReLU) vs the simplex control.

In [ ]:
m_result = run_mlp_phase2(
    mlp, mlp_config.p,
    target_r2=0.99, max_freqs=10, n_pcs=30, ridge_alpha=1e-3,
)
summarize_phase2(m_result)

In [ ]:
save_fig(plot_phase2_summary(m_result), 'phase2_summary.png');

In [ ]:
# Class-mean summary: kills per-sample noise that drowns the cross-c
# Fourier structure in the raw scatter. M1 colored by a, M2 by c.
save_fig(plot_phase2_summary_classmean(m_result), 'phase2_summary_classmean.png');

In [ ]:
save_fig(plot_m_per_freq_grid(m_result), 'phase2_per_freq_grid.png');

In [ ]:
# M2 (post-ReLU) — analogue of Poncini Fig 3
# M2 needs class-mean to integrate out per-(a,b) noise from post-ReLU frequency mixing.
save_fig(plot_m_per_freq_pairs(m_result, 'M2', class_mean='c'), 'phase2_per_freq_pairs_m2.png');

In [ ]:
# M1 (pre-ReLU) — analogue of Poncini Fig 5 (sum-of-circles target)
save_fig(plot_m_per_freq_pairs(m_result, 'M1', n_pcs=40), 'phase2_per_freq_pairs_m1.png');

## Phase 3 — Cross-architecture comparison

Headline questions:
- Do both architectures hit R² ≈ 0.99 against the EHMM and ≈ 0.09 against the
  HMM-simplex control? (Same algorithm family.)
- Same Fourier modes or different? (Strong vs weak universality.)

In [ ]:
summarize_phase3(t_result, m_result)

In [ ]:
save_fig(plot_alpha_comparison(t_result, m_result, transformer_config.p), 'phase3_alpha_comparison.png');

In [ ]:
save_fig(plot_decoding_comparison(t_result, m_result), 'phase3_decoding_comparison.png');

## Training trajectory — Poncini Fig 2

For each saved checkpoint, decode the EHMM polygon (using the (ω_k, α_k) fitted
once at the final checkpoint) and the HMM simplex (one-hot per c). Default
position is post-modular-sum: POS4 for the transformer, M2 for the MLP. Expected
shape:

- **Polygon (EHMM, red)**: starts near 0, climbs sharply during the grokking
  transition, plateaus near 1 once the model has converged.
- **Simplex (HMM, blue)**: bumps slightly during memorisation (model has
  some marginal mass on c) but stays around 0.1 — the model never learns the
  HMM-style representation.

Sweep takes ~1–3 minutes per architecture on a laptop GPU (200 checkpoints,
one forward pass + two ridge fits each).

In [ ]:
t_sweep = sweep_transformer('transformer_run', run_root=RUN_ROOT, device=DEVICE)

In [ ]:
m_sweep = sweep_mlp('mlp_run', run_root=RUN_ROOT, device=DEVICE)

In [ ]:
save_fig(plot_trajectory_comparison(t_sweep, m_sweep), 'trajectory_comparison.png');

### Pre-MLP comparison (POS3 ↔ M1) — apples-to-apples cross-architecture decoding

The post-MLP comparison above (POS4 vs M2) sits immediately before each model's
unembed, where activations are dominated by the model's prediction of c. For
the MLP this inflates the simplex R² substantially: M2 has d_hidden=512
dimensions of capacity and no residual stream, so it ends up encoding
prediction-like one-hot structure as well as the EHMM polygon.

A fairer architectural comparison is at the pre-MLP layer in each model — the
position where the additive structure ⟨v(a)| + ⟨v(b)| has formed but the
modular-sum non-linearity hasn't yet collapsed it onto a single c:

- Transformer: **POS3** (post-attention, pre-MLP-block residual stream)
- MLP        : **M1**   (pre-ReLU sum of left/right embeddings)

At these positions, the EHMM target is ⟨v_ω(a)| + ⟨v_ω(b)| (sum-of-two-circles)
and we expect cleaner separation from the simplex control in *both*
architectures, since neither has yet committed to a c.

Note: at POS3 / M1 the simplex target ⟨e_p(a+b mod p)| is *not* a natural
representation for the activations to encode — the model hasn't computed
(a+b) yet at that layer — so the simplex curve here is a more honest
"unrelated control" than at POS4 / M2.

In [ ]:
t_sweep_pre = sweep_transformer('transformer_run', position='POS3',
                                run_root=RUN_ROOT, device=DEVICE)

In [ ]:
m_sweep_pre = sweep_mlp('mlp_run', position='M1',
                        run_root=RUN_ROOT, device=DEVICE)

In [ ]:
save_fig(plot_trajectory_comparison(t_sweep_pre, m_sweep_pre),
         'trajectory_comparison_pre_mlp.png');